# Team 08 - Leekhith: Agent B (PPO view refiner) + integration + team results
Runtime: CPU is fine. Run Step 1 in every new session.

Needs `team08` in YOUR My Drive (Shared with me -> right-click team08 -> Organize -> Add shortcut -> My Drive).

## Step 1 - Mount, unpack, install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/team08'
!mkdir -p $DRIVE/results
!rm -rf /content/uav_inspection
!unzip -o -q $DRIVE/uav_inspection.zip -d /content/
%cd /content/uav_inspection
!pip install -q gymnasium stable-baselines3 tensorboard
!ls scripts

## Step 2 - Smoke test + baselines for your mode
Look at the `refine` block: random / fixed_orbit fail, scripted_approach is an upper bound.

In [ ]:
!python scripts/test_env.py | tail -1
!python scripts/run_baselines.py --out $DRIVE/results

# PART 1 - Agent B (your viva component)
## Step 3 - Train Agent B with PPO (~2-4 min)

In [ ]:
!python scripts/train_agent_b.py --steps 100000 --out $DRIVE/agentB

## Step 4 - Learning curves + results table

In [ ]:
import glob, json, numpy as np, pandas as pd, matplotlib.pyplot as plt
df = pd.concat([pd.read_csv(f, skiprows=1) for f in glob.glob(f'{DRIVE}/agentB/monitor/*.monitor.csv')]).sort_values('t')
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(df['r'].rolling(200).mean().values); ax[0].set_title('Episode return (rolling 200)'); ax[0].set_xlabel('episode')
ax[1].plot(df['success'].astype(float).rolling(200).mean().values); ax[1].set_title('Success rate (best view reached)')
ax[2].plot(df['l'].rolling(200).mean().values); ax[2].set_title('Steps to best view')
plt.tight_layout(); plt.savefig(f'{DRIVE}/agentB/learning_curves.png', dpi=150); plt.show()
pd.read_csv(f'{DRIVE}/agentB/agentB_vs_baselines.csv')[['policy','return_mean','success_mean','steps_mean','energy_used_mean','train_minutes']].round(3)

## Step 5 - Live demo: state -> Gaussian policy -> action -> reward -> next state (CO3)

In [ ]:
import torch
from stable_baselines3 import PPO
from env import InspectionEnv
agent = PPO.load(f'{DRIVE}/agentB/best/best_model.zip', device='cpu')
env = InspectionEnv('refine'); obs, info = env.reset(seed=1000)
NAMES = ['distance', 'angle', 'lateral', 'target_u', 'battery', 'time_left']
for t in range(8):
    print(f'--- t={t} state:', {n: round(float(v), 2) for n, v in zip(NAMES, obs)}, f'view quality s={env.s:.2f}')
    with torch.no_grad():
        dist = agent.policy.get_distribution(agent.policy.obs_to_tensor(obs)[0]).distribution
    mu, sd = dist.mean[0].numpy(), dist.stddev[0].numpy()
    print('   policy N(mu, sigma): mu =', np.round(mu, 2), ' sigma =', np.round(sd, 2))
    obs, r, term, trunc, info = env.step(mu)
    print(f'   ACTION [d_dist, d_angle, d_lateral] = {np.round(np.clip(mu, -1, 1), 2)}  REWARD = {r:+.3f}')
    if term or trunc: print('   -> best view reached (q=4), episode ends'); break

## Step 6 - Convergence + sample efficiency (CO4)

In [ ]:
succ = df['success'].astype(float).rolling(200).mean().values; cum = df['l'].cumsum().values
conv = int(np.argmax(succ >= 0.95)) if (succ >= 0.95).any() else None
print(f'Episodes: {len(df)}, env steps: {int(cum[-1]):,}')
if conv is not None: print(f'Convergence (success >= 95%): episode {conv} = {int(cum[conv]):,} env steps')
ev = np.load(f'{DRIVE}/agentB/eval/evaluations.npz')
plt.figure(figsize=(6, 4)); plt.plot(ev['timesteps'], ev['results'].mean(1)); plt.title('Evaluation return vs timesteps')
plt.xlabel('timesteps'); plt.tight_layout(); plt.savefig(f'{DRIVE}/agentB/eval_curve.png', dpi=150); plt.show()

## Step 7 - Exploration sensitivity: entropy coefficient + clip range (CO5, ~10 min)

In [ ]:
for ent in [0.0, 0.01, 0.05]:
    !python scripts/train_agent_b.py --steps 60000 --ent_coef {ent} --out $DRIVE/agentB_ent{ent}
for clip in [0.1, 0.3]:
    !python scripts/train_agent_b.py --steps 60000 --clip {clip} --out $DRIVE/agentB_clip{clip}

In [ ]:
rows = []
for tag in ['ent0.0', 'ent0.01', 'ent0.05', 'clip0.1', 'clip0.3']:
    r = pd.read_csv(f'{DRIVE}/agentB_{tag}/agentB_vs_baselines.csv').iloc[0]
    d = pd.concat([pd.read_csv(f, skiprows=1) for f in glob.glob(f'{DRIVE}/agentB_{tag}/monitor/*.monitor.csv')]).sort_values('t')
    s = d['success'].astype(float).rolling(200).mean().values; c = d['l'].cumsum().values
    conv = int(c[np.argmax(s >= 0.95)]) if (s >= 0.95).any() else None
    rows.append({'run': tag, 'return': round(r.return_mean, 2), 'success': r.success_mean, 'steps': r.steps_mean, 'env_steps_to_95%': conv})
pd.DataFrame(rows)

# PART 2 - Integration (after Yeseswini's Agent A and Mukhesh's Agent C exist in Drive)
## Step 8 - Run all three agents together (team demo + ablation)

In [ ]:
!python scripts/run_full_mode.py --a $DRIVE/agentA/best/best_model.zip --b $DRIVE/agentB/best/best_model.zip \
     --c $DRIVE/agentC/best/best_model.zip --out $DRIVE/full

## Step 9 - Team results summary (for team slides + team poster)

In [ ]:
def first_row(path, label):
    r = pd.read_csv(path).iloc[0]; return {'component': label, 'return': round(r.return_mean, 2), 'success': r.success_mean,
                                             'steps': r.steps_mean, 'energy': round(r.energy_used_mean, 3)}
team = [first_row(f'{DRIVE}/agentA/agentA_vs_baselines.csv', 'Agent A - DQN navigation'),
        first_row(f'{DRIVE}/agentB/agentB_vs_baselines.csv', 'Agent B - PPO refinement'),
        first_row(f'{DRIVE}/agentC/agentC_vs_baselines.csv', 'Agent C - A2C supervisor')]
display(pd.DataFrame(team))
full = pd.read_csv(f'{DRIVE}/full/full_mode_results.csv')
display(full[['config','return_mean','success_mean','coverage_mean','defects_found_mean','miou_mean','energy_used_mean']].round(3))
fig, ax = plt.subplots(1, 2, figsize=(12, 4)); names = [c.split('(')[0].strip() for c in full['config']]
ax[0].barh(names, full['energy_used_mean']); ax[0].set_title('Energy used per mission (lower = better)')
ax[1].barh(names, full['defects_found_mean']); ax[1].set_title('Defects found (of 26)')
plt.tight_layout(); plt.savefig(f'{DRIVE}/full/team_summary.png', dpi=150); plt.show()
print(open(f'{DRIVE}/full/demo_mission.txt').read())

## Step 10 (optional) - Agent B on real-image uncertainty (after Mukhesh's cache exists)

In [ ]:
!python scripts/train_agent_b.py --detector learned --cache $DRIVE/detector_cache.npz --out $DRIVE/agentB_learned